# Model Evaluation — 2024 Team wRC+ Projections

Compares all 15 regression × classification model combinations against actual 2024 team wRC+.

**Metrics**
- **RMSE** and **MAE** — computed on model-mean predictions (p1) or the single deterministic estimate (p2)
- **80% predictive coverage** — fraction of teams whose actual wRC+ falls inside [p10, p90]; **p1 only** (p2 has no interval)

A positive Δ(p1 − p2) for RMSE/MAE means the stochastic procedure is *worse*; for coverage
a positive value for `cov80_dist_p1` means p1 is farther from the nominal 80%.

In [9]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import re
from pathlib import Path

data_path = Path.cwd()
while not (data_path / "data" / "fangraphs" / "fg_hitters.csv").exists() and data_path != data_path.parent:
    data_path = data_path.parent
DATA_DIR = data_path

sim   = pd.read_csv(DATA_DIR / "data" / "model_outputs" / "model_team_sim_2024.csv", index_col="team")
actuals = pd.read_csv(DATA_DIR / "data" / "fangraphs" / "fg_team_wrc_2024_2025.csv")

actual_2024 = (
    actuals[actuals["Season"] == 2024]
    .set_index("Team")["wRC+"]
    .rename("actual_wRC+")
)

# Align: keep only teams present in both sources
common = sim.index.intersection(actual_2024.index)
sim_aligned    = sim.loc[common]
actual_aligned = actual_2024.loc[common]

print(f"Simulation teams  : {len(sim.index)}")
print(f"Actual teams      : {len(actual_2024)}")
print(f"Common (used)     : {len(common)}")
print(f"\nMissing from sim  : {sorted(set(actual_2024.index) - set(sim.index))}")
print(f"Missing from actual: {sorted(set(sim.index) - set(actual_2024.index))}")


Simulation teams  : 30
Actual teams      : 30
Common (used)     : 30

Missing from sim  : []
Missing from actual: []


In [10]:
# Parse column names:
#   p1: {(reg, cls, 'p1'): {'mean': col, 'p10': col, 'p90': col, ...}}
#   p2: {(reg, cls, 'p2'): {'wRC+': col}}
combo_cols = {}
for col in sim.columns:
    m1 = re.match(r'^(.+?)_(lr_m\d+)_(p1)_(mean|median|p10|p90)$', col)
    if m1:
        reg, cls, proc, stat = m1.groups()
        combo_cols.setdefault((reg, cls, proc), {})[stat] = col
        continue
    m2 = re.match(r'^(.+?)_(lr_m\d+)_(p2)_wRC\+$', col)
    if m2:
        reg, cls, proc = m2.groups()
        combo_cols.setdefault((reg, cls, proc), {})['wRC+'] = col

combos = sorted(combo_cols.keys())
print(f"Parsed {len(combos)} (reg, cls, proc) combinations:")
for k in combos:
    print(f"  {k}  →  cols: {list(combo_cols[k].keys())}")

Parsed 30 (reg, cls, proc) combinations:
  ('BBK_Brl_homo', 'lr_m02', 'p1')  →  cols: ['mean', 'median', 'p10', 'p90']
  ('BBK_Brl_homo', 'lr_m02', 'p2')  →  cols: ['wRC+']
  ('BBK_Brl_homo', 'lr_m07', 'p1')  →  cols: ['mean', 'median', 'p10', 'p90']
  ('BBK_Brl_homo', 'lr_m07', 'p2')  →  cols: ['wRC+']
  ('BBK_Brl_homo', 'lr_m09', 'p1')  →  cols: ['mean', 'median', 'p10', 'p90']
  ('BBK_Brl_homo', 'lr_m09', 'p2')  →  cols: ['wRC+']
  ('BMC', 'lr_m02', 'p1')  →  cols: ['mean', 'median', 'p10', 'p90']
  ('BMC', 'lr_m02', 'p2')  →  cols: ['wRC+']
  ('BMC', 'lr_m07', 'p1')  →  cols: ['mean', 'median', 'p10', 'p90']
  ('BMC', 'lr_m07', 'p2')  →  cols: ['wRC+']
  ('BMC', 'lr_m09', 'p1')  →  cols: ['mean', 'median', 'p10', 'p90']
  ('BMC', 'lr_m09', 'p2')  →  cols: ['wRC+']
  ('BMC_theta', 'lr_m02', 'p1')  →  cols: ['mean', 'median', 'p10', 'p90']
  ('BMC_theta', 'lr_m02', 'p2')  →  cols: ['wRC+']
  ('BMC_theta', 'lr_m07', 'p1')  →  cols: ['mean', 'median', 'p10', 'p90']
  ('BMC_theta', 'lr_

In [11]:
def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))

def mae(pred, actual):
    return float(np.mean(np.abs(pred - actual)))

def coverage_80(lo, hi, actual):
    return float(np.mean((actual >= lo) & (actual <= hi)))


rows = []
actual_vals = actual_aligned.values

for reg, cls, proc in combos:
    cols = combo_cols[(reg, cls, proc)]

    if proc == 'p1':
        mean_ = sim_aligned[cols['mean']].values
        lo_   = sim_aligned[cols['p10']].values
        hi_   = sim_aligned[cols['p90']].values
        cov   = round(coverage_80(lo_, hi_, actual_vals), 3)
    else:  # p2 — single deterministic estimate, no interval
        mean_ = sim_aligned[cols['wRC+']].values
        cov   = float('nan')

    rows.append({
        'reg_model':   reg,
        'cls_model':   cls,
        'proc':        proc,
        'rmse':        round(rmse(mean_, actual_vals), 3),
        'mae':         round(mae(mean_, actual_vals), 3),
        'coverage_80': cov,
    })

long_df = pd.DataFrame(rows)
print(long_df.to_string(index=False))

     reg_model cls_model proc   rmse   mae  coverage_80
  BBK_Brl_homo    lr_m02   p1 10.061 8.139        0.633
  BBK_Brl_homo    lr_m02   p2 10.246 8.244          NaN
  BBK_Brl_homo    lr_m07   p1  9.977 8.009        0.633
  BBK_Brl_homo    lr_m07   p2 10.294 8.306          NaN
  BBK_Brl_homo    lr_m09   p1 10.069 8.062        0.667
  BBK_Brl_homo    lr_m09   p2 10.288 8.298          NaN
           BMC    lr_m02   p1  8.558 7.237        0.567
           BMC    lr_m02   p2  8.381 7.108          NaN
           BMC    lr_m07   p1  8.489 7.066        0.533
           BMC    lr_m07   p2  8.351 6.970          NaN
           BMC    lr_m09   p1  8.467 7.058        0.600
           BMC    lr_m09   p2  8.384 7.007          NaN
     BMC_theta    lr_m02   p1  8.501 7.089        0.300
     BMC_theta    lr_m02   p2  8.371 7.102          NaN
     BMC_theta    lr_m07   p1  8.568 7.070        0.267
     BMC_theta    lr_m07   p2  8.356 6.973          NaN
     BMC_theta    lr_m09   p1  8.529 7.025      

In [12]:
from itertools import product as iproduct

out_rows = []
reg_models = sorted(long_df['reg_model'].unique())
cls_models = sorted(long_df['cls_model'].unique())

for reg, cls in iproduct(reg_models, cls_models):
    p1 = long_df[(long_df['reg_model'] == reg) & (long_df['cls_model'] == cls) & (long_df['proc'] == 'p1')].iloc[0]
    p2 = long_df[(long_df['reg_model'] == reg) & (long_df['cls_model'] == cls) & (long_df['proc'] == 'p2')].iloc[0]

    out_rows.append({
        'reg_model':          reg,
        'cls_model':          cls,
        # RMSE
        'rmse_p1':            p1['rmse'],
        'rmse_p2':            p2['rmse'],
        'rmse_delta_p1_p2':   round(p1['rmse'] - p2['rmse'], 3),
        # MAE
        'mae_p1':             p1['mae'],
        'mae_p2':             p2['mae'],
        'mae_delta_p1_p2':    round(p1['mae'] - p2['mae'], 3),
        # 80% coverage — p1 only (p2 has no predictive interval)
        'cov80_p1':           p1['coverage_80'],
        'cov80_dist_p1':      round(abs(p1['coverage_80'] - 0.80), 3),
    })

results_df = pd.DataFrame(out_rows)

out_path = DATA_DIR / "data" / "evaluation" / "model_comparison_2024.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(f"  {results_df.shape[0]} rows × {results_df.shape[1]} columns")
print()
print(results_df.to_string(index=False))

Saved → /Users/johannhatzius/Desktop/GitHub/personal/classes/LSE/ST451-Bayesian_ML/project/data/evaluation/model_comparison_2024.csv
  15 rows × 10 columns

     reg_model cls_model  rmse_p1  rmse_p2  rmse_delta_p1_p2  mae_p1  mae_p2  mae_delta_p1_p2  cov80_p1  cov80_dist_p1
  BBK_Brl_homo    lr_m02   10.061   10.246            -0.185   8.139   8.244           -0.105     0.633          0.167
  BBK_Brl_homo    lr_m07    9.977   10.294            -0.317   8.009   8.306           -0.297     0.633          0.167
  BBK_Brl_homo    lr_m09   10.069   10.288            -0.219   8.062   8.298           -0.236     0.667          0.133
           BMC    lr_m02    8.558    8.381             0.177   7.237   7.108            0.129     0.567          0.233
           BMC    lr_m07    8.489    8.351             0.138   7.066   6.970            0.096     0.533          0.267
           BMC    lr_m09    8.467    8.384             0.083   7.058   7.007            0.051     0.600          0.200
     BMC_t